# Ethiopia Financial Inclusion Data Exploration and Enrichment

## Task 1: Data Exploration and Enrichment

This notebook performs comprehensive data exploration and enrichment for Ethiopia's financial inclusion forecasting project.

### Objectives:
1. Load and explore the unified dataset
2. Explain the unified schema and record types
3. Analyze trends and identify gaps
4. Enrich the dataset with new contextual data
5. Document all additions

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')  # Use default instead of seaborn-v0_8 for compatibility
sns.set_palette("husl")

# Define data paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load and Explore the Data

In [2]:
# Load the main dataset
try:
    df_main = pd.read_excel(DATA_RAW / 'ethiopia_fi_unified_data.xlsx')
    print(f"Main dataset loaded successfully: {df_main.shape}")
except Exception as e:
    print(f"Error loading main dataset: {e}")
    # Create a sample dataset for demonstration
    df_main = pd.DataFrame({
        'record_id': [1, 2, 3, 4],
        'record_type': ['observation', 'observation', 'event', 'target'],
        'indicator_code': ['FI_ACCESS_ACCOUNT', 'FI_USAGE_DIGITAL_PAY', None, 'FI_ACCESS_ACCOUNT'],
        'year': [2017, 2017, 2018, 2025],
        'value': [34.8, 8.9, None, 55.0]
    })
    print("Using sample dataset for demonstration")

# Load reference codes
try:
    df_ref = pd.read_excel(DATA_RAW / 'reference_codes.xlsx')
    print(f"Reference codes loaded successfully: {df_ref.shape}")
except Exception as e:
    print(f"Error loading reference codes: {e}")
    # Use the CSV version we created earlier
    try:
        df_ref = pd.read_csv('../reference_codes.csv')
        print(f"Reference codes loaded from CSV: {df_ref.shape}")
    except:
        print("Could not load reference codes from any source")
        df_ref = pd.DataFrame()

print("\nDatasets loaded!")

Main dataset loaded successfully: (43, 34)
Reference codes loaded successfully: (71, 4)

Datasets loaded!


In [3]:
# Inspect the structure of the main dataset
print("=== MAIN DATASET STRUCTURE ===")
print(f"Shape: {df_main.shape}")
print(f"\nColumns: {list(df_main.columns)}")
print(f"\nData types:")
print(df_main.dtypes)
print(f"\nFirst 5 rows:")
display(df_main.head())

=== MAIN DATASET STRUCTURE ===
Shape: (43, 34)

Columns: ['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']

Data types:
record_id                      object
record_type                    object
category                       object
pillar                         object
indicator                      object
indicator_code                 object
indicator_direction            object
value_numeric                 float64
value_text                     object
value_type                     object
unit                 

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


In [4]:
# Inspect reference codes
print("=== REFERENCE CODES STRUCTURE ===")
if not df_ref.empty:
    print(f"Shape: {df_ref.shape}")
    print(f"\nColumns: {list(df_ref.columns)}")
    
    # Check for code_type column or similar
    if 'code_type' in df_ref.columns:
        print(f"\nCode types available:")
        print(df_ref['code_type'].value_counts())
    else:
        print(f"\nNo 'code_type' column found. Available columns: {list(df_ref.columns)}")
        # Try to find similar columns
        type_cols = [col for col in df_ref.columns if 'type' in col.lower()]
        if type_cols:
            print(f"Found type-related columns: {type_cols}")
            print(df_ref[type_cols[0]].value_counts())
    
    display(df_ref.head(10))
else:
    print("Reference codes dataset is empty or not loaded")

=== REFERENCE CODES STRUCTURE ===
Shape: (71, 4)

Columns: ['field', 'code', 'description', 'applies_to']

No 'code_type' column found. Available columns: ['field', 'code', 'description', 'applies_to']


,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All
5,record_type,forecast,Predicted future value,All
6,category,product_launch,New product or service introduced,event
7,category,market_entry,New competitor enters market,event
8,category,market_exit,Competitor leaves market,event
9,category,policy,Government strategy or regulatory framework,event


## 2. Explain the Unified Schema

The unified schema is designed to capture both quantitative financial inclusion data and the contextual events that drive changes. Here's how each record type works:

In [5]:
# Analyze record types in the dataset
print("=== RECORD TYPE ANALYSIS ===")
if 'record_type' in df_main.columns:
    record_type_counts = df_main['record_type'].value_counts()
    print(f"Record type distribution:")
    print(record_type_counts)
    
    # Show examples of each record type
    for record_type in df_main['record_type'].unique():
        print(f"\n--- {record_type.upper()} RECORDS ---")
        sample = df_main[df_main['record_type'] == record_type].head(2)
        
        # Select only columns that exist in the dataframe
        desired_cols = ['record_id', 'record_type', 'parent_id', 'indicator_code', 'year', 'value', 'event_type', 'event_description']
        existing_cols = [col for col in desired_cols if col in df_main.columns]
        
        if existing_cols:
            display(sample[existing_cols])
        else:
            print("Available columns:", list(df_main.columns))
            display(sample)
else:
    print("No 'record_type' column found in the dataset")
    print("Available columns:", list(df_main.columns))
    display(df_main.head())

=== RECORD TYPE ANALYSIS ===
Record type distribution:
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

--- OBSERVATION RECORDS ---


,record_id,record_type,indicator_code
0,REC_0001,observation,ACC_OWNERSHIP
1,REC_0002,observation,ACC_OWNERSHIP



--- TARGET RECORDS ---


,record_id,record_type,indicator_code
30,REC_0031,target,ACC_OWNERSHIP
31,REC_0032,target,ACC_FAYDA



--- EVENT RECORDS ---


,record_id,record_type,indicator_code
33,EVT_0001,event,EVT_TELEBIRR
34,EVT_0002,event,EVT_SAFARICOM


### Schema Explanation:

1. **observation**: Actual measured data points from surveys (Global Findex) or administrative sources
   - Contains `indicator_code`, `year`, `value`, and demographic breakdowns
   - Forms the foundation for trend analysis and forecasting

2. **event**: Policy changes, infrastructure developments, or strategic initiatives
   - Contains `event_type`, `event_description`, `year`
   - Provides contextual drivers that may influence financial inclusion

3. **impact_link**: Connects events to indicators showing causal relationships
   - Uses `parent_id` to reference the event record
   - Links to specific `indicator_code` to show which metrics are affected
   - Contains `impact_description` explaining the expected relationship

4. **target**: Future projections or policy targets
   - Similar structure to observations but for future years
   - Used for validation and goal-setting

In [6]:
# Demonstrate impact_link relationships
print("=== IMPACT_LINK RELATIONSHIPS ===")
if 'record_type' in df_main.columns:
    impact_links = df_main[df_main['record_type'] == 'impact_link']
    
    if not impact_links.empty:
        for _, link in impact_links.iterrows():
            if 'parent_id' in df_main.columns:
                parent_event = df_main[df_main['record_id'] == link['parent_id']]
                if not parent_event.empty:
                    event_desc = parent_event.iloc[0].get('event_description', 'Unknown event')
                    print(f"Event: {event_desc}")
                    print(f"  → Impacts: {link.get('indicator_code', 'Unknown indicator')}")
                    print(f"  → Impact: {link.get('impact_description', 'No description')}")
                    print()
    else:
        print("No impact_link records found in current dataset - this is a gap we'll address in enrichment!")
else:
    print("Cannot analyze impact links without record_type column")

=== IMPACT_LINK RELATIONSHIPS ===
No impact_link records found in current dataset - this is a gap we'll address in enrichment!


## 3. Data Exploration - Trends and Gaps Analysis

In [7]:
# Analyze Access and Usage indicators
if 'record_type' in df_main.columns:
    observations = df_main[df_main['record_type'] == 'observation'].copy()
else:
    # If no record_type, assume all are observations
    observations = df_main.copy()

print("=== INDICATOR ANALYSIS ===")
if 'indicator_code' in observations.columns:
    print(f"Available indicators:")
    print(observations['indicator_code'].value_counts())
else:
    print("No indicator_code column found")

if 'year' in observations.columns:
    print(f"\nYear range: {observations['year'].min()} - {observations['year'].max()}")
    print(f"Years available: {sorted(observations['year'].unique())}")
else:
    print("No year column found")

print(f"\nDemographic breakdowns available:")
demographic_cols = ['gender', 'age_group', 'region', 'income_level']
for col in demographic_cols:
    if col in observations.columns:
        print(f"{col}: {observations[col].unique()}")
    else:
        print(f"{col}: Column not found")

=== INDICATOR ANALYSIS ===
Available indicators:
indicator_code
ACC_OWNERSHIP         6
ACC_FAYDA             3
ACC_MM_ACCOUNT        2
ACC_4G_COV            2
USG_P2P_COUNT         2
GEN_GAP_ACC           2
ACC_MOBILE_PEN        1
USG_ATM_COUNT         1
USG_ATM_VALUE         1
USG_CROSSOVER         1
USG_P2P_VALUE         1
USG_TELEBIRR_USERS    1
USG_TELEBIRR_VALUE    1
USG_MPESA_ACTIVE      1
USG_MPESA_USERS       1
USG_ACTIVE_RATE       1
AFF_DATA_INCOME       1
GEN_MM_SHARE          1
GEN_GAP_MOBILE        1
Name: count, dtype: int64
No year column found

Demographic breakdowns available:
gender: ['all' 'male' 'female']
age_group: Column not found
region: [nan]
income_level: Column not found


In [8]:
# Create visualizations if we have the necessary columns
if all(col in observations.columns for col in ['year', 'value', 'indicator_code']):
    # Visualize trends for key indicators
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Ethiopia Financial Inclusion Trends', fontsize=16, fontweight='bold')
    
    # Filter for national level data if possible
    if 'gender' in observations.columns and 'region' in observations.columns:
        national_data = observations[
            (observations['gender'] == 'All') & 
            (observations['region'] == 'National')
        ]
    else:
        national_data = observations
    
    # Account Access trend
    access_data = national_data[national_data['indicator_code'] == 'FI_ACCESS_ACCOUNT']
    if not access_data.empty:
        axes[0,0].plot(access_data['year'], access_data['value'], marker='o', linewidth=2, markersize=8)
        axes[0,0].set_title('Account Access (National)', fontweight='bold')
        axes[0,0].set_ylabel('Percentage (%)')
        axes[0,0].grid(True, alpha=0.3)
    
    # Digital Payment Usage trend
    usage_data = national_data[national_data['indicator_code'] == 'FI_USAGE_DIGITAL_PAY']
    if not usage_data.empty:
        axes[0,1].plot(usage_data['year'], usage_data['value'], marker='o', linewidth=2, markersize=8, color='orange')
        axes[0,1].set_title('Digital Payment Usage (National)', fontweight='bold')
        axes[0,1].set_ylabel('Percentage (%)')
        axes[0,1].grid(True, alpha=0.3)
    
    # Gender comparison if gender column exists
    if 'gender' in observations.columns:
        # Gender comparison for Account Access
        gender_access = observations[
            (observations['indicator_code'] == 'FI_ACCESS_ACCOUNT') & 
            (observations['gender'].isin(['Male', 'Female']))
        ]
        
        for gender in ['Male', 'Female']:
            data = gender_access[gender_access['gender'] == gender]
            if not data.empty:
                axes[1,0].plot(data['year'], data['value'], marker='o', label=gender, linewidth=2, markersize=8)
        
        axes[1,0].set_title('Account Access by Gender', fontweight='bold')
        axes[1,0].set_ylabel('Percentage (%)')
        axes[1,0].set_xlabel('Year')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
        
        # Gender comparison for Digital Payment Usage
        gender_usage = observations[
            (observations['indicator_code'] == 'FI_USAGE_DIGITAL_PAY') & 
            (observations['gender'].isin(['Male', 'Female']))
        ]
        
        for gender in ['Male', 'Female']:
            data = gender_usage[gender_usage['gender'] == gender]
            if not data.empty:
                axes[1,1].plot(data['year'], data['value'], marker='o', label=gender, linewidth=2, markersize=8)
        
        axes[1,1].set_title('Digital Payment Usage by Gender', fontweight='bold')
        axes[1,1].set_ylabel('Percentage (%)')
        axes[1,1].set_xlabel('Year')
        axes[1,1].legend()
        axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot create visualizations - missing required columns (year, value, indicator_code)")
    print(f"Available columns: {list(observations.columns)}")

Cannot create visualizations - missing required columns (year, value, indicator_code)
Available columns: ['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']


In [9]:
# Identify data gaps and missing contextual drivers
print("=== DATA GAPS ANALYSIS ===")

# Check for missing demographic breakdowns
print("1. DEMOGRAPHIC COVERAGE GAPS:")
demographic_cols = ['age_group', 'region', 'income_level', 'education_level']
for col in demographic_cols:
    if col in observations.columns:
        non_all_count = len(observations[observations[col] != 'All']) if 'All' in observations[col].values else len(observations)
        print(f"   - {col} breakdowns: {non_all_count} records")
    else:
        print(f"   - {col}: Column not found")

# Check temporal gaps
if 'year' in observations.columns:
    print("\n2. TEMPORAL GAPS:")
    years_available = sorted(observations['year'].unique())
    print(f"   - Available years: {years_available}")
    if len(years_available) > 1:
        print(f"   - Gap between surveys: {years_available[-1] - years_available[0]} years")
    print(f"   - Missing intermediate years for trend analysis")

# Check contextual events
if 'record_type' in df_main.columns:
    events = df_main[df_main['record_type'] == 'event']
    print(f"\n3. CONTEXTUAL EVENTS:")
    print(f"   - Total events recorded: {len(events)}")
    if not events.empty and 'event_type' in events.columns:
        print(f"   - Event types: {events['event_type'].value_counts().to_dict()}")
        if 'year' in events.columns:
            print(f"   - Event years: {sorted(events['year'].unique())}")
    
    # Check impact links
    impact_links = df_main[df_main['record_type'] == 'impact_link']
    print(f"\n4. CAUSAL RELATIONSHIPS:")
    print(f"   - Impact links recorded: {len(impact_links)}")
    if len(impact_links) == 0:
        print(f"   - This is a critical gap for forecasting models!")
else:
    print("\nCannot analyze events and impact links without record_type column")

=== DATA GAPS ANALYSIS ===
1. DEMOGRAPHIC COVERAGE GAPS:
   - age_group: Column not found
   - region breakdowns: 30 records
   - income_level: Column not found
   - education_level: Column not found

3. CONTEXTUAL EVENTS:
   - Total events recorded: 10

4. CAUSAL RELATIONSHIPS:
   - Impact links recorded: 0
   - This is a critical gap for forecasting models!


## Summary of Current Data Structure

Based on the exploration above, we can see what data is available and what needs to be enriched. The next steps would be to:

1. **Standardize the schema** if columns are missing
2. **Add missing record types** (events, impact_links, targets)
3. **Enrich demographic breakdowns** 
4. **Add contextual events and causal relationships**

This analysis provides the foundation for the data enrichment process.

In [10]:
# Display final summary
print("=== FINAL DATA SUMMARY ===")
print(f"Main dataset shape: {df_main.shape}")
print(f"Main dataset columns: {list(df_main.columns)}")
if not df_ref.empty:
    print(f"Reference codes shape: {df_ref.shape}")
    print(f"Reference codes columns: {list(df_ref.columns)}")

print("\n=== NEXT STEPS ===")
print("1. Verify column names match expected schema")
print("2. Add missing record types if needed")
print("3. Proceed with data enrichment")
print("4. Create comprehensive documentation")

=== FINAL DATA SUMMARY ===
Main dataset shape: (43, 34)
Main dataset columns: ['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']
Reference codes shape: (71, 4)
Reference codes columns: ['field', 'code', 'description', 'applies_to']

=== NEXT STEPS ===
1. Verify column names match expected schema
2. Add missing record types if needed
3. Proceed with data enrichment
4. Create comprehensive documentation
